# Jane Street Market Forecasting - LGBM Baseline (Offline, Memory-Aware)

This notebook builds a simple offline LightGBM baseline for `responder_6` with a focus on running on limited RAM:

- **Polars-first** pipeline (lazy scan + feature engineering in Polars)
- Aggressive but safe dtype downcasting (`float32`, compact ints)
- Convert to pandas **only per fold** right before training
- Time-series-aware 3-fold CV with each validation fold close to **6 months** of `date_id`
- Competition metric: **sample-weighted zero-mean R^2**

This is intentionally for local offline iteration only.

In [1]:
# Uncomment and run once if needed.
# !pip install lightgbm polars pyarrow pandas scikit-learn

import gc
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "jane-street-market-forecasting":
    PROJECT_DIR = PROJECT_DIR / "jane-street-market-forecasting"

DATA_DIR = PROJECT_DIR / "data"
TRAIN_DIR = DATA_DIR / "train.parquet"
TRAIN_GLOB = str(TRAIN_DIR / "partition_id=*" / "*.parquet")

TARGET = "responder_6"
WEIGHT_COL = "weight"
FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)]
BASE_COLS = ["date_id", "time_id", "symbol_id", WEIGHT_COL, TARGET] + FEATURE_COLS

N_FOLDS = 1
DATE_GAP = 1
VAL_DAYS = 180  # roughly 6 months by date_id
TRAIN_LOOKBACK_DAYS = 540  # cap train window for memory; set None for full expanding window

# Optional cap for quick experiments / memory safety.
# Keep >= (N_FOLDS * VAL_DAYS + 180) for meaningful folds.
MAX_DATES = 730

print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"TRAIN_DIR exists: {TRAIN_DIR.exists()}")

DATA_DIR exists: True
TRAIN_DIR exists: True


In [3]:
def build_scan() -> pl.LazyFrame:
    scan = pl.scan_parquet(TRAIN_GLOB).select(BASE_COLS)

    if MAX_DATES is not None:
        max_date = scan.select(pl.max("date_id").alias("max_date")).collect().item()
        min_date = max_date - MAX_DATES + 1
        scan = scan.filter(pl.col("date_id") >= min_date)

    # Downcast to reduce memory pressure while preserving practical precision.
    cast_exprs = [
        pl.col("date_id").cast(pl.Int16),
        pl.col("time_id").cast(pl.Int16),
        pl.col("symbol_id").cast(pl.Int16),
        pl.col(WEIGHT_COL).cast(pl.Float32),
        pl.col(TARGET).cast(pl.Float32),
    ] + [pl.col(c).cast(pl.Float32) for c in FEATURE_COLS]

    scan = scan.with_columns(cast_exprs)
    return scan


scan = build_scan()
df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

print(df_pl.shape)
print(df_pl.select(["date_id", "time_id", "symbol_id", WEIGHT_COL, TARGET]).head())
print(f"Estimated in-memory size (MB): {df_pl.estimated_size('mb'):.2f}")

(26659688, 84)
shape: (5, 5)
┌─────────┬─────────┬───────────┬──────────┬─────────────┐
│ date_id ┆ time_id ┆ symbol_id ┆ weight   ┆ responder_6 │
│ ---     ┆ ---     ┆ ---       ┆ ---      ┆ ---         │
│ i16     ┆ i16     ┆ i16       ┆ f32      ┆ f32         │
╞═════════╪═════════╪═══════════╪══════════╪═════════════╡
│ 969     ┆ 0       ┆ 0         ┆ 3.269281 ┆ 1.121171    │
│ 969     ┆ 0       ┆ 1         ┆ 6.936615 ┆ 1.147567    │
│ 969     ┆ 0       ┆ 2         ┆ 2.6898   ┆ 0.023757    │
│ 969     ┆ 0       ┆ 3         ┆ 2.604316 ┆ -0.091458   │
│ 969     ┆ 0       ┆ 4         ┆ 2.394183 ┆ -4.300847   │
└─────────┴─────────┴───────────┴──────────┴─────────────┘
Estimated in-memory size (MB): 8498.19


In [4]:
def add_basic_features_polars(frame: pl.DataFrame) -> pl.DataFrame:
    max_time = max(int(frame["time_id"].max()), 1)
    feature_exprs = [pl.col(c) for c in FEATURE_COLS]

    out = frame.with_columns(
        [
            pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int16) for c in FEATURE_COLS])
            .cast(pl.Int16)
            .alias("feature_nan_count"),
            pl.mean_horizontal(feature_exprs).cast(pl.Float32).alias("feature_row_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in FEATURE_COLS])
            .cast(pl.Float32)
            .alias("feature_row_abs_mean"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .sin()
            .cast(pl.Float32)
            .alias("time_sin"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .cos()
            .cast(pl.Float32)
            .alias("time_cos"),
        ]
    )
    return out


def weighted_zero_mean_r2(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true**2))
    if den == 0:
        return np.nan
    return 1.0 - num / den


def make_time_series_folds(
    unique_dates: np.ndarray,
    n_folds: int = 3,
    val_days: int = 180,
    gap: int = 1,
    train_lookback_days: int | None = 540,
):
    """
    Build chronological folds with fixed-size validation blocks near the dataset tail.
    Each fold validation span is `val_days` (about 6 months).
    """
    n_dates = len(unique_dates)
    required = n_folds * val_days + gap + 30
    if n_dates < required:
        raise ValueError(
            f"Not enough dates ({n_dates}) for {n_folds} folds with val_days={val_days}."
        )

    folds = []
    for k in range(n_folds):
        # Older fold first, newest fold last
        val_end = n_dates - (n_folds - 1 - k) * val_days
        val_start = val_end - val_days
        train_end = val_start - gap

        if train_lookback_days is None:
            train_start = 0
        else:
            train_start = max(0, train_end - train_lookback_days)

        train_dates = unique_dates[train_start:train_end]
        val_dates = unique_dates[val_start:val_end]

        if len(train_dates) == 0 or len(val_dates) == 0:
            raise ValueError("Empty train/val split created. Increase MAX_DATES.")

        folds.append((train_dates, val_dates))

    return folds


df_fe_pl = add_basic_features_polars(df_pl)
MODEL_FEATURES = [
    c
    for c in df_fe_pl.columns
    if c not in {TARGET, WEIGHT_COL} and not c.startswith("responder_")
]

print(f"Model features: {len(MODEL_FEATURES)}")
print(f"Feature-engineered size (MB): {df_fe_pl.estimated_size('mb'):.2f}")

Model features: 87
Feature-engineered size (MB): 8955.84


In [6]:
unique_dates = np.sort(df_fe_pl["date_id"].unique().to_numpy())
folds = make_time_series_folds(
    unique_dates,
    n_folds=N_FOLDS,
    val_days=VAL_DAYS,
    gap=DATE_GAP,
    train_lookback_days=TRAIN_LOOKBACK_DAYS,
)

for i, (tr_dates, va_dates) in enumerate(folds, start=1):
    print(
        f"Fold {i}: train [{int(tr_dates.min())}, {int(tr_dates.max())}] ({len(tr_dates)} dates) | "
        f"val [{int(va_dates.min())}, {int(va_dates.max())}] ({len(va_dates)} dates, ~6 months)"
    )

Fold 1: train [978, 1517] (540 dates) | val [1519, 1698] (180 dates, ~6 months)


In [8]:
params = {
    "objective": "regression",
    "learning_rate": 0.05,
    "n_estimators": 800,
    "num_leaves": 64,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
}

fold_metrics = []
global_val_num = 0.0
global_val_den = 0.0

for fold_idx, (train_dates, val_dates) in enumerate(folds, start=1):
    train_pl = df_fe_pl.filter(pl.col("date_id").is_in(train_dates))
    val_pl = df_fe_pl.filter(pl.col("date_id").is_in(val_dates))

    medians = train_pl.select([pl.col(c).median().alias(c) for c in MODEL_FEATURES]).row(
        0, named=True
    )
    fill_exprs = [
        pl.col(c)
        .fill_null(medians[c])
        .fill_nan(medians[c])
        .cast(pl.Float32)
        .alias(c)
        for c in MODEL_FEATURES
    ]

    train_pl = train_pl.with_columns(fill_exprs)
    val_pl = val_pl.with_columns(fill_exprs)

    print(
        f"Fold {fold_idx} rows | train={train_pl.height:,}, val={val_pl.height:,} | "
        f"train_mb~{train_pl.estimated_size('mb'):.1f}, val_mb~{val_pl.estimated_size('mb'):.1f}"
    )

    X_train = train_pl.select(MODEL_FEATURES).to_pandas()
    y_train = train_pl[TARGET].to_numpy().astype(np.float32)
    w_train = train_pl[WEIGHT_COL].to_numpy().astype(np.float32)

    X_val = val_pl.select(MODEL_FEATURES).to_pandas()
    y_val = val_pl[TARGET].to_numpy().astype(np.float32)
    w_val = val_pl[WEIGHT_COL].to_numpy().astype(np.float32)

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train,
        y_train,
        sample_weight=w_train,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[w_val],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    train_pred = model.predict(X_train, num_iteration=model.best_iteration_)
    val_pred = model.predict(X_val, num_iteration=model.best_iteration_)

    train_r2 = weighted_zero_mean_r2(y_train, train_pred, w_train)
    val_r2 = weighted_zero_mean_r2(y_val, val_pred, w_val)

    val_num = float(np.sum(w_val * (y_val - val_pred) ** 2))
    val_den = float(np.sum(w_val * (y_val**2)))
    global_val_num += val_num
    global_val_den += val_den

    fold_metrics.append(
        {
            "fold": fold_idx,
            "best_iteration": int(model.best_iteration_ or params["n_estimators"]),
            "train_rows": int(train_pl.height),
            "val_rows": int(val_pl.height),
            "train_weighted_zero_mean_r2": train_r2,
            "val_weighted_zero_mean_r2": val_r2,
        }
    )

    print(
        f"Fold {fold_idx} | best_iter={fold_metrics[-1]['best_iteration']} | "
        f"train_r2={train_r2:.6f} | val_r2={val_r2:.6f}"
    )

    del train_pl, val_pl, X_train, X_val, y_train, y_val, w_train, w_val, train_pred, val_pred, model
    gc.collect()

metrics_df = pd.DataFrame(fold_metrics)
metrics_df

Fold 1 rows | train=19,639,752, val=6,686,944 | train_mb~6731.1, val_mb~2293.7
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.387545 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21021
[LightGBM] [Info] Number of data points in the train set: 19639752, number of used features: 87
[LightGBM] [Info] Start training from score -0.001029
Fold 1 | best_iter=227 | train_r2=0.044386 | val_r2=0.009124


,fold,best_iteration,train_rows,val_rows,train_weighted_zero_mean_r2,val_weighted_zero_mean_r2
0,1,227,19639752,6686944,0.044386,0.009124


In [9]:
overall_oof_r2 = np.nan if global_val_den == 0 else 1.0 - (global_val_num / global_val_den)

print("\nTrain/Eval metrics summary")
print(metrics_df.to_string(index=False))
print(f"\nMean fold train weighted zero-mean R^2: {metrics_df['train_weighted_zero_mean_r2'].mean():.6f}")
print(f"Mean fold val weighted zero-mean R^2: {metrics_df['val_weighted_zero_mean_r2'].mean():.6f}")
print(f"Overall OOF weighted zero-mean R^2: {overall_oof_r2:.6f}")


Train/Eval metrics summary
 fold  best_iteration  train_rows  val_rows  train_weighted_zero_mean_r2  val_weighted_zero_mean_r2
    1             227    19639752   6686944                     0.044386                   0.009124

Mean fold train weighted zero-mean R^2: 0.044386
Mean fold val weighted zero-mean R^2: 0.009124
Overall OOF weighted zero-mean R^2: 0.009124
